# 05 · 数据管理与模型训练
使用模块四的真实 MuJoCo episode，完成质量检查、版本快照、按采样分组划分、官方 ACT 网络训练与闭环评估，以及 DINO-WM 的视觉预测和反事实动作推演。

DINOv2 预训练权重首次从 Meta 官方地址下载并转为 safetensors；后续复用本工作区的不可变权重。ACT 采用 LeRobot 官方网络，DINO-WM 使用官方因果 ViT 核心，并明确记录残差预测头、小型 RGB 解码器和示例分辨率等适配。示例训练结果不代表论文评测成绩。

In [ ]:
import json
import os
import sys
import uuid
from pathlib import Path

os.environ.setdefault("MUJOCO_GL", "egl")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("OMP_NUM_THREADS", "4")
PROJECT_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/physrealx").is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if Path(sys.prefix).name != "PhysReal-X":
    raise RuntimeError("请选择 PhysReal-X Conda 环境的 Jupyter 内核")
from examples._common import (
    NotebookRun,
    operation_receipt,
    output_artifact,
    require_completed,
)

from physrealx import ArtifactRef, PhysRealX, RunSpec
from physrealx.domain import TrainingSpec
from physrealx.integrations.datasets import (
    dataset_bundle,
    load_snapshot,
)

px = PhysRealX.open(PROJECT_ROOT / ".physrealx")
OUTPUT_DIR = PROJECT_ROOT / "examples/data_management_and_training/output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
run = NotebookRun(
    OUTPUT_DIR / "report.json",
    (
        "input",
        "ingest",
        "snapshot",
        "policy_train",
        "policy_validation",
        "world_train",
        "rollout",
        "counterfactual",
        "long_rollout",
        "export",
    ),
    {"python": sys.executable, "conda_prefix": sys.prefix},
    evidence_scope="trained_models_on_grouped_simulator_episodes",
)
run_key = uuid.uuid4().hex[:12]
print(sys.executable)

In [ ]:
with run.stage("input") as stage:
    upstream = require_completed(
        PROJECT_ROOT / "examples/simulation_and_data_generation/output/report.json"
    )
    collection = ArtifactRef.model_validate(upstream["stages"]["quality"]["collection"])
    evaluation_run = RunSpec.model_validate(upstream["stages"]["expert"]["run"]["spec"])
    stage["collection"] = collection

In [ ]:
with run.stage("ingest") as stage:
    operation = px.datasets.ingest(
        collection, method="episode_dataset", config={"resource_id": f"cabinet2.dataset.{run_key}"}
    )
    dataset_artifact = output_artifact(operation, "dataset")
    dataset = dataset_bundle(px.artifacts, dataset_artifact)
    stage["operation"] = operation_receipt(operation)
    print(dict(operation.metrics))

In [ ]:
with run.stage("snapshot") as stage:
    operation = px.datasets.snapshot(
        dataset_artifact, method="episode_split", config={"seed": 2026}
    )
    snapshot_ref = output_artifact(operation, "snapshot")
    snapshot, _, partitions = load_snapshot(px.artifacts, snapshot_ref)
    train_ref = px.datasets.training_subset(
        snapshot_ref,
        resource_id=f"cabinet2.expert.train.{run_key}",
        required_tag="expert_demonstration",
    )
    stage["operation"] = operation_receipt(operation)
    stage["snapshot"] = snapshot_ref
    stage["train_dataset"] = train_ref
    print({name: len(values) for name, values in partitions.items()})

In [ ]:
with run.stage("policy_train") as stage:
    import torch

    from physrealx.integrations.act import ACTTrainer

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_ref = px.training.submit(
        TrainingSpec(
            resource_id=f"cabinet2.act.{run_key}",
            version="1.0.0",
            dataset=train_ref,
            trainer=ACTTrainer.descriptor.bind(
                config={
                    "steps": 300,
                    "device": device,
                    "seed": 2026,
                    "observation_signals": [
                        "joint.position",
                        "joint.velocity",
                        "joint.cabinet.gui_drawer_2.joint.position",
                        "joint.cabinet.gui_drawer_2.joint.velocity",
                    ],
                }
            ),
        )
    )
    model = px.training.get_model(model_ref)
    stage["model"] = model_ref
    stage["metrics"] = model.metrics
    print(device, model.metrics)

In [ ]:
with run.stage("policy_validation") as stage:
    evaluation_ref = px.evaluations.submit(
        resource_id=f"cabinet2.act.validation.{run_key}",
        version="1.0.0",
        model=model_ref,
        run=evaluation_run,
        episode_count=5,
        base_seed=91000,
    )
    evaluation = px.evaluations.get_report(evaluation_ref)
    # Record the initial policy honestly; final policy acceptance follows feedback in module six.
    assert evaluation.metrics["episode_count"] == 5
    stage["evaluation"] = evaluation_ref
    stage["success_rate"] = evaluation.success_rate
    print("独立闭环成功率:", evaluation.success_rate)

In [ ]:
with run.stage("world_train") as stage:
    import urllib.request
    from tempfile import TemporaryDirectory

    from physrealx.integrations.world_model import (
        DINO_WEIGHTS_URL,
        EncoderSpec,
        import_dinov2_weights,
    )

    cache = PROJECT_ROOT / "examples/data_management_and_training/input/dinov2-encoder.json"
    if cache.is_file() and px.artifacts.verify(
        EncoderSpec.model_validate_json(cache.read_text()).weights
    ):
        encoder_ref = px.artifacts.put_json(
            EncoderSpec.model_validate_json(cache.read_text()).model_dump(
                mode="json", by_alias=True
            )
        )
    else:
        with TemporaryDirectory(prefix="physrealx-dinov2-") as directory:
            weights_path = Path(directory) / "dinov2_vits14.pth"
            with urllib.request.urlopen(
                DINO_WEIGHTS_URL, timeout=60
            ) as response, weights_path.open("wb") as target:
                size = 0
                while chunk := response.read(1024 * 1024):
                    size += len(chunk)
                    if size > 128 * 1024**2:
                        raise ValueError("DINOv2 下载超出预设大小限制")
                    target.write(chunk)
            encoder_ref = import_dinov2_weights(weights_path, px.artifacts)
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_bytes(px.artifacts.get_bytes(encoder_ref))
    operation = px.world_models.train(
        snapshot_ref,
        encoder_ref,
        method="dino_wm",
        config={
            "device": device,
            "steps": 1800,
            "history": 1,
            "training_horizon": 16,
            "evaluation_horizon": 100,
            "validation_interval": 200,
            "state_loss_weight": 5,
            "state_signals": [
                "joint.position",
                "joint.velocity",
                "joint.cabinet.gui_drawer_2.joint.position",
                "joint.cabinet.gui_drawer_2.joint.velocity",
            ],
        },
    )
    for metric in ("normalized_state_rmse", "latent_mse", "rgb_mse"):
        assert (
            operation.metrics["test_" + metric] < operation.metrics["test_persistence_" + metric]
        ), metric
    stage["quality_gate"] = "state_latent_rgb_beat_identical_context_persistence"
    world_model_ref = output_artifact(operation, "model")
    world_report_ref = output_artifact(operation, "training_report")
    stage["operation"] = operation_receipt(operation)
    stage["model"] = world_model_ref
    print(dict(operation.metrics))

In [ ]:
with run.stage("rollout") as stage:
    import numpy as np

    from physrealx.integrations._artifacts import read_array
    from physrealx.integrations.world_model import WorldModelActions, WorldModelContext

    test_episode = next(episode for episode in partitions["test"] if episode.success)
    test_summary = next(
        row for row in dataset.episodes if row.episode_id == test_episode.episode_id
    )
    start_index, horizon = 0, 12
    context_ref = px.artifacts.put_json(
        WorldModelContext(episode=test_summary, start_index=start_index).model_dump(
            mode="json", by_alias=True
        )
    )
    future_actions = [
        step.action.inline for step in test_episode.steps[start_index : start_index + horizon]
    ]
    interval = test_episode.steps[1].timestamp_seconds - test_episode.steps[0].timestamp_seconds
    actions_ref = px.artifacts.put_json(
        WorldModelActions(values=future_actions, interval_seconds=interval).model_dump(
            mode="json", by_alias=True
        )
    )
    operation = px.world_models.rollout(
        world_model_ref, context_ref, actions_ref, method="dino_wm", config={"device": device}
    )
    rollout = json.loads(px.artifacts.get_bytes(output_artifact(operation, "rollout")))
    predicted = read_array(px.artifacts, ArtifactRef.model_validate(rollout["states"]))
    future_rgb = read_array(px.artifacts, ArtifactRef.model_validate(rollout["rgb"]))
    truth = np.array(
        [
            step.observations["joint.position"].inline[0]
            for step in test_episode.steps[start_index + 1 : start_index + horizon + 1]
        ]
    )
    position_rmse_m = float(np.sqrt(np.mean((predicted[:, 0] - truth) ** 2)))
    assert np.isfinite(predicted).all() and future_rgb.shape[0] == horizon
    stage["operation"] = operation_receipt(operation)
    stage["position_rmse_m"] = position_rmse_m
    print("未参与训练 episode 的预测位置 RMSE (m):", position_rmse_m)

In [ ]:
with run.stage("counterfactual") as stage:
    import matplotlib.pyplot as plt

    from physrealx import MethodCapability

    changed_actions = [[0.0] for _ in range(100)]
    changed_ref = px.artifacts.put_json(
        WorldModelActions(values=changed_actions, interval_seconds=interval)
    )
    operation = px.world_models.rollout(
        world_model_ref, context_ref, changed_ref, method="dino_wm", config={"device": device}
    )
    changed = json.loads(px.artifacts.get_bytes(output_artifact(operation, "rollout")))
    counterfactual = read_array(px.artifacts, ArtifactRef.model_validate(changed["states"]))
    # Reuse the exact scene, task, backend and seed. Only future robot actions change.
    counter_run = px.runs.submit(
        RunSpec(
            scene=test_episode.scene,
            task=test_episode.task,
            backend=test_episode.backend,
            policy=px.methods.bind(
                MethodCapability.POLICY_INFERENCE,
                "joint_sequence",
                config={"commands": changed_actions},
            ),
            episode_count=1,
            base_seed=test_episode.seed,
            stop_on_success=False,
            dataset_id=f"cabinet2.counterfactual.{run_key}",
            dataset_version="1.0.0",
            record_signals=evaluation_run.record_signals,
            tags=("counterfactual_truth", test_episode.episode_id),
        )
    )
    counter_episode = px.runs.episodes(counter_run)[0]
    state_names = (
        "joint.position",
        "joint.velocity",
        "joint.cabinet.gui_drawer_2.joint.position",
        "joint.cabinet.gui_drawer_2.joint.velocity",
    )
    counter_truth = np.array(
        [
            [s.observations[name].inline[0] for name in state_names]
            for s in counter_episode.steps[1:101]
        ]
    )
    counter_rmse = np.sqrt(np.mean((counterfactual - counter_truth) ** 2, axis=0))
    assert counter_rmse[[0, 2]].max() < 0.03, counter_rmse
    stage["operation"] = operation_receipt(operation)
    stage["truth_dataset"] = counter_run.result.dataset
    stage["state_rmse"] = counter_rmse.tolist()
    stage["action_sensitivity"] = float(np.max(np.abs(counterfactual[:horizon] - predicted)))
    print("五秒反事实的真实引擎对照 RMSE:", counter_rmse)

## 单帧条件与五秒未来推演
世界模型配置 history=1，只使用起点的一帧 RGB 和同步状态。下方生成 100 步、共五秒的动作条件未来图像与状态；超出已记录轨迹的部分用于展示推演接口，不能计入有真值的预测精度。


In [ ]:
with run.stage("long_rollout") as stage:
    five_second_steps = round(5.0 / interval)
    assert len(test_episode.steps) > five_second_steps
    continuation = [s.action.inline for s in test_episode.steps[:five_second_steps]]
    long_actions = px.artifacts.put_json(
        WorldModelActions(values=continuation, interval_seconds=interval)
    )
    operation = px.world_models.rollout(
        world_model_ref, context_ref, long_actions, method="dino_wm", config={"device": device}
    )
    prediction = json.loads(px.artifacts.get_bytes(output_artifact(operation, "rollout")))
    long_states = read_array(px.artifacts, ArtifactRef.model_validate(prediction["states"]))
    long_truth = np.array(
        [
            [s.observations[name].inline[0] for name in state_names]
            for s in test_episode.steps[1 : five_second_steps + 1]
        ]
    )
    initial_state = np.array(
        [test_episode.steps[0].observations[name].inline[0] for name in state_names]
    )
    rmse = np.sqrt(np.mean((long_states - long_truth) ** 2, axis=0))
    baseline_rmse = np.sqrt(np.mean((initial_state - long_truth) ** 2, axis=0))
    assert np.isfinite(long_states).all() and rmse[[0, 2]].max() < 0.03, rmse
    assert np.linalg.norm(rmse[[0, 2]]) < np.linalg.norm(baseline_rmse[[0, 2]]), (
        rmse,
        baseline_rmse,
    )
    stage["operation"] = operation_receipt(operation)
    stage["context_frames"] = 1
    stage["prediction_duration_seconds"] = five_second_steps * interval
    stage["state_signals"] = list(state_names)
    stage["state_rmse"] = rmse.tolist()
    stage["persistence_rmse"] = baseline_rmse.tolist()
    stage["evaluation_scope"] = (
        "five-second held-out simulator truth and identical-state persistence baseline"
    )
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for axis, index, name in zip(axes, (0, 2), ("Robot pusher", "Passive drawer")):
        axis.plot(long_truth[:, index], label="MuJoCo truth")
        axis.plot(long_states[:, index], label="Predicted")
        axis.plot(counterfactual[:, index], label="Counterfactual prediction")
        axis.plot(counter_truth[:, index], linestyle="--", label="Counterfactual truth")
        axis.set_title(name)
        axis.set_ylabel("Position / m")
        axis.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "world-model.png", dpi=140)
    plt.show()
    print("五秒状态 RMSE:", rmse, "；保持原状态基线:", baseline_rmse)

In [ ]:
with run.stage("export") as stage:
    stage["model"] = model_ref
    stage["world_model"] = world_model_ref
    stage["train_dataset"] = train_ref
    stage["snapshot"] = snapshot_ref
    stage["dataset"] = dataset_artifact
    stage["evaluation_run"] = evaluation_run
    stage["failure_dataset"] = upstream["stages"]["export"]["failure_dataset"]
    for filename, ref in (
        ("world-model.json", world_model_ref),
        ("world-model-training.json", world_report_ref),
    ):
        (OUTPUT_DIR / filename).write_bytes(px.artifacts.get_bytes(ref))
run.complete()
print("模块五完成：分组数据、ACT 闭环评测、DINO-WM 实训与反事实预测。")